# Policy gradients (REINFORCE)

_Modern Deep Learning & AI_

**Skip the value table. Directly raise the chance of actions that led to a big reward.**

DQN learns values, then acts greedily. Policy gradients skip that. They learn the policy itself: the probability of each action.

     The rule is wonderfully direct: if an action led to a high reward, make it more likely next time. If it led to a low reward, make it less likely.

---

This notebook builds REINFORCE one step at a time. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

### Step 1 — Define the policy network

The policy is a small neural net that maps a state to **logits** over the available actions. We do not learn values here — the net directly parameterizes the action probabilities. We pair it with an Adam optimizer that will nudge the weights.

In [ ]:
import torch
import torch.nn as nn
from torch.distributions import Categorical

class Policy(nn.Module):
    def __init__(self, state_dim, n_actions):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 32),
            nn.ReLU(),
            nn.Linear(32, n_actions),       # logits over actions
        )

    def forward(self, s):
        return self.net(s)

policy = Policy(state_dim=4, n_actions=2)
opt = torch.optim.Adam(policy.parameters(), lr=1e-2)

### Step 2 — Make a synthetic episode and its returns

We fake one short episode: 5 visited states, each with a per-step return `G` (the total future reward from that step). We **standardize** the returns by subtracting the mean and dividing by the std — this acts as a baseline that cuts variance and tells us which steps were better or worse than average.

In [ ]:
# synthetic episode: 5 states visited, with a per-step return G
states  = torch.randn(5, 4)
returns = torch.tensor([2.0, 1.5, 1.0, -0.5, -1.0])

# standardize returns: a simple baseline that reduces variance
returns = (returns - returns.mean()) / (returns.std() + 1e-8)

### Step 3 — Sample actions and take the REINFORCE step

We turn the logits into a categorical distribution, sample the actions actually taken, and record their log-probabilities `log pi(a | s)`. The REINFORCE loss is `-(log_prob * return)` summed over the episode: minimizing it raises the probability of actions with positive return and lowers it for negative return. One backward pass and optimizer step applies the update.

In [ ]:
logits = policy(states)
dist = Categorical(logits=logits)

actions = dist.sample()                # actions actually taken
log_probs = dist.log_prob(actions)     # log pi(a | s)

loss = -(log_probs * returns).sum()    # REINFORCE objective

opt.zero_grad()
loss.backward()
opt.step()

print(float(loss))

## Visualize the data & results

_On the same corridor task, does a REINFORCE policy learn? Does the return G grow as we train?_

### Step 1 — Set up the corridor task and policy

The environment is an 8-state corridor: the agent starts at state 0 and wants to reach the goal at state 7, where it gets `+10` (every other step costs `-0.1`). The policy is a table of softmax logits `theta`, one row per state with two action preferences (move left or right).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(1)

N, GOAL = 8, 7                  # 8-state corridor, goal reward = +10
theta = np.zeros((N, 2))       # softmax policy logits per state
lr, gamma = 0.05, 0.95

returns_hist = []

### Step 2 — Roll out an episode under the current policy

For each episode we walk the corridor for up to 30 steps. At each state we convert that state's logits to action probabilities with a softmax, sample an action, move, and collect the reward. We record the full trajectory of `(state, action, reward)` tuples and stop early if we reach the goal.

In [ ]:
def run_episode():
    s = 0
    traj = []
    G = 0.0
    for _ in range(30):
        p = np.exp(theta[s] - theta[s].max())     # softmax pi(.|s)
        p /= p.sum()
        a = rng.choice(2, p=p)

        s2 = min(N - 1, s + 1) if a == 1 else max(0, s - 1)   # right or left
        r = 10.0 if s2 == GOAL else -0.1

        traj.append((s, a, r))
        G += r
        s = s2
        if s2 == GOAL:
            break
    return traj, G

### Step 3 — Apply the REINFORCE update over 300 episodes

We walk each trajectory **backwards** to accumulate the discounted return `Gt` at each step. The score-function gradient of the softmax is `grad = (one-hot action) - p`, and we nudge `theta` by `lr * (Gt - baseline) * grad`. Subtracting the mean-reward baseline reduces variance so learning is stabler. We log the episode return each time.

In [ ]:
for ep in range(300):
    traj, G = run_episode()

    Gt = 0.0
    baseline = G / len(traj)              # mean-reward baseline cuts variance
    for s, a, r in reversed(traj):        # walk trajectory backwards
        Gt = r + gamma * Gt

        p = np.exp(theta[s] - theta[s].max())
        p /= p.sum()

        grad = -p
        grad[a] += 1.0                    # d log pi / d theta
        theta[s] += lr * (Gt - baseline) * grad

    returns_hist.append(G)

### Step 4 — Plot the learning curve

We smooth the per-episode returns with a 20-episode moving average and plot them. As REINFORCE makes rewarding actions more likely, the smoothed return `G` should climb and then plateau once the policy reliably reaches the goal.

In [ ]:
sm = np.convolve(returns_hist, np.ones(20) / 20, mode='valid')

plt.figure(figsize=(6, 4))
plt.plot(sm, color='#7ee787', label='return G')
plt.xlabel('episode')
plt.ylabel('smoothed return G')
plt.title('REINFORCE on 8-state corridor')
plt.legend()
plt.tight_layout()
plt.show()